# LLMOps with Agent-Based RAG and MLflow Tracing

**LLM Security & Safety Research Papers - Intelligent Agent-Based Document Analysis**

---

This notebook demonstrates a complete **LLMOps lifecycle** using an intelligent agent.

# Problem Statement

## Business Context

Organizations deploying LLM-based systems face a rapidly evolving landscape of security threats, safety risks,
and compliance requirements. Research papers on LLM security are dense and fragmented across multiple domains.
A simple RAG pipeline can retrieve passages, but security analysts need to **search**, **summarize**,
and **reason** across multiple papers. An **agent** can replicate this workflow by deciding which tool to use
for each question.

## Objective

Build an **intelligent LLMOps agent** that:
1. Processes multiple LLM security research papers through a data pipeline (loading, chunking, embedding)
2. Uses **3 tools** (document search, summarization, reasoning) to answer security research questions
3. Is **versioned** (prompt versions tracked in MLflow)
4. Is **monitored** (MLflow 3.10.x autolog traces every tool call, LLM invocation, and latency)
5. Is **evaluated** (RAGAS metrics) and collects **human feedback** (RLHF pattern)
6. Is **orchestrated** by Prefect for reproducible workflows

## Data Description

A corpus of **4 research papers** on LLM security and safety:

| Paper | Topic | Pages |
|---|---|---|
| Pankajakshan et al. (2024) | Mapping LLM Security Landscapes - OWASP risk assessment for LLMs | ~15 |
| Greshake et al. (2023) | Indirect Prompt Injection attacks on LLM-integrated applications | ~30 |
| Chen et al. (2025) | AI Safety Landscape - Trustworthy, Responsible, and Safe AI framework | ~96 |
| Inan et al. (2023) | Llama Guard - LLM-based input-output safeguard model | ~12 |

# Installing and Importing Libraries

We install all required libraries. Key upgrades from a basic RAG notebook:
- **`mlflow==3.10.1`** - Latest version with `langchain.autolog()` for automatic agent tracing
- **`langchain_openai`** - Modern LangChain OpenAI integration
- **`langchain.agents`** - Agent framework with tool-use capabilities

In [ ]:
# ============================================================================
# INSTALL REQUIRED LIBRARIES
# ============================================================================
# mlflow==3.10.1        -> Experiment tracking with automatic agent tracing
# langchain + openai    -> Agent framework with tool-use and LLM integration
# chromadb              -> Vector database for embedding storage
# pymupdf               -> PDF document loading
# ragas                 -> RAG evaluation metrics (faithfulness, relevancy)
# prefect               -> Workflow orchestration (tracks pipeline runs)
# ============================================================================

!pip install -q langchain_community==0.3.27 \
              langchain==0.3.27 \
              langchain_openai==0.3.27 \
              chromadb==1.0.15 \
              pymupdf==1.26.3 \
              tiktoken==0.9.0 \
              ragas==0.3.0 \
              datasets==4.0.0 \
              evaluate==0.4.5 \
              mlflow[skinny]==3.10.1 \
              prefect==3.4.22 \
              openai==1.109.1

**Note**: After running the above cell, **restart the runtime** (Runtime -> Restart runtime)
before running the cells below. This ensures all packages are properly loaded.

In [ ]:
# ============================================================================
# IMPORTS
# ============================================================================

# --- Standard library ---
import os
import json
import time
import random
import subprocess
from pathlib import Path

# --- PDF and text processing ---
from langchain_community.document_loaders import PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
import tiktoken
import pandas as pd

# --- Embeddings and vector store ---
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma

# --- Agent framework ---
# These are the key imports that differentiate an agent from a simple RAG chain:
# - create_openai_tools_agent: Creates an agent that uses OpenAI function calling
#   to decide which tool to invoke at each step
# - AgentExecutor: Wraps the agent with error handling, iteration limits, and
#   intermediate step tracking
# - Tool: Defines a capability the agent can use (search, summarize, reason)
from langchain.agents import AgentExecutor, create_openai_tools_agent
from langchain.tools import Tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# --- Evaluation ---
from ragas import evaluate as ragas_evaluate
from ragas.metrics import Faithfulness, AnswerRelevancy
from datasets import Dataset

# --- LLMOps: Orchestration + Tracking ---
from prefect import flow, task          # Workflow orchestration
import mlflow                            # Experiment tracking & tracing
import mlflow.langchain                  # LangChain-specific autolog
from openai import OpenAI                # Direct API client

print("All imports successful!")
print(f"MLflow version: {mlflow.__version__}")
from langchain_community.callbacks import get_openai_callback


# Configuration & MLflow Setup

## OpenAI Setup

> **Prerequisite**: Store your API credentials in Colab's **Secrets** panel
> (key icon in the left sidebar):
> - `OPENAI_API_KEY` - Authentication key for the OpenAI-compatible API
> - `OPENAI_API_BASE` - Base URL for the API endpoint
>
> Alternatively, upload a `config_llmops.json` file to `/content/` as a fallback.
>
> **Note**: MLflow UI is exposed via a free Cloudflare Tunnel (no auth token needed).

In [ ]:
# ============================================================================
# LOAD API CREDENTIALS
# ============================================================================
# Primary: Colab Secrets (secure, no file uploads needed)
# Fallback: secrets.json file in /content/
# ============================================================================

from google.colab import userdata

def get_secret(name):
    """Try Colab Secrets first, fall back to config JSON."""
    try:
        return userdata.get(name)
    except Exception:
        pass
    # Fallback to config file
    try:
        with open("secrets.json", "r") as f:
            return json.load(f).get(name)
    except Exception:
        raise ValueError(f"Secret '{name}' not found in Colab Secrets or secrets.json")

OPENAI_API_KEY = get_secret("OPENAI_API_KEY")
OPENAI_API_BASE = get_secret("OPENAI_API_BASE")

# Store credentials in environment variables (required by LangChain and OpenAI)
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["OPENAI_BASE_URL"] = OPENAI_API_BASE

# Initialize the LLM that will power our agent
# ChatOpenAI is the LangChain wrapper around the OpenAI chat completion API.
# gpt-5-mini uses default temperature (1) as it does not support custom values.
llm = ChatOpenAI(
    model="gpt-5-mini",
    openai_api_key=OPENAI_API_KEY,
    openai_api_base=OPENAI_API_BASE
)

print("API credentials loaded and LLM initialized.")

## MLflow Setup with Automatic Agent Tracing

**MLflow 3.10.x** introduces `mlflow.langchain.autolog()` which automatically traces
LangChain agent executions. Every tool call, every LLM invocation, every reasoning step
is captured as a **trace span** - with zero manual instrumentation.

This is the **Model Monitoring** component of the LLMOps architecture.

In [ ]:
# ============================================================================
# MLFLOW EXPERIMENT TRACKING SETUP
# ============================================================================
# MLflow stores experiment data (parameters, metrics, artifacts, traces) on disk.
# In production, this would point to a remote tracking server.
# ============================================================================

# Directory where MLflow will persist all experiment logs
mlruns_path = Path("/content/llmops_mlflow")

# Set tracking URI - tells MLflow where to store data
mlflow.set_tracking_uri(f"file:///{mlruns_path.resolve()}")
print("MLflow tracking URI:", mlflow.get_tracking_uri())

# Create an experiment to group all our runs
mlflow.set_experiment("LLMOps_Agent_Demo")

# ============================================================================
# ENABLE AUTOMATIC AGENT TRACING (MLflow 3.10.x feature)
# ============================================================================
# This single line captures:
#   - Agent reasoning steps (which tool to call and why)
#   - Each tool invocation with inputs and outputs
#   - Each LLM call with prompt, response, token count, and latency
#   - The full execution tree as a hierarchical trace
#
# Without autolog, you would need to manually instrument every function.
# ============================================================================

mlflow.langchain.autolog(
    log_traces=True              # Capture full agent execution traces
)

print("MLflow autolog enabled - all agent traces will be captured automatically.")

# Data Processing Pipeline

**Architecture mapping**: `Proprietary Data -> Data Processing Pipeline -> Embeddings (Vector Storage)`

This section implements the top portion of the LLMOps architecture diagram.
We take unstructured PDF data and transform it into machine-searchable embeddings.

### LLMOps vs MLOps Challenge

| Parameter | MLOps | LLMOps (What We Do Here) |
|---|---|---|
| **Data Type** | Structured/tabular | Unstructured (PDF text) |
| **Feature Engineering** | Handcrafted features | Tokenization + embeddings |
| **Data Drift** | Distribution shifts | Semantic drift (language evolution) |

In [ ]:
# Discover all PDFs in the folder (our "proprietary data" in the architecture)
import glob as glob_mod

# Auto-detect environment: Colab uses /content/PDFs/, local repo uses LLMOps/PDFs/
if os.path.isdir("/content/PDFs"):
    pdf_dir = "/content/PDFs/"
elif os.path.isdir("PDFs"):
    pdf_dir = "PDFs/"
elif os.path.isdir("LLMOps/PDFs"):
    pdf_dir = "LLMOps/PDFs/"
else:
    raise FileNotFoundError("PDFs directory not found. Place PDFs in /content/PDFs/ (Colab) or LLMOps/PDFs/ (local).")

pdf_files = sorted(glob_mod.glob(pdf_dir + "*.pdf"))

print(f"Found {len(pdf_files)} PDFs in {pdf_dir}:")
for f in pdf_files:
    print(f"  - {Path(f).name}")

## Document Loading and Chunking

We split the PDF into overlapping chunks of ~512 tokens each. This is the
LLMOps equivalent of **feature engineering**:
- **Chunk size** controls how much context each retrieval returns
- **Chunk overlap** prevents losing information at chunk boundaries
- **Tokenizer** (cl100k_base) matches OpenAI's tokenizer for accurate sizing

In [ ]:
# ============================================================================
# TASK: Load and Split PDFs into Chunks
# ============================================================================
# The @task decorator (from Prefect) makes this function a tracked unit of work.
# Prefect logs the task's start time, duration, success/failure, and inputs.
# This is part of the LLMOps "Data Processing Pipeline".
# ============================================================================

@task
def load_and_split_pdfs(pdf_paths):
    """Load multiple PDF documents and split them into chunks for embedding.

    LLMOps concept: Data Processing Pipeline
    - Converts unstructured PDF data into structured text chunks
    - Uses tiktoken cl100k_base encoding (matches OpenAI's tokenizer)
    - chunk_size=512 balances context richness vs retrieval precision
    - chunk_overlap=20 prevents information loss at boundaries
    """
    all_docs = []
    for pdf_path in pdf_paths:
        loader = PyMuPDFLoader(pdf_path)
        docs = loader.load()
        all_docs.extend(docs)
        print(f"  Loaded {len(docs)} pages from {Path(pdf_path).name}")

    # Clean special tokens from PDF text that can break tiktoken encoding
    for doc in all_docs:
        doc.page_content = doc.page_content.replace('<|endoftext|>', ' ')

    # RecursiveCharacterTextSplitter intelligently splits text at paragraph,
    # sentence, and word boundaries to keep chunks semantically coherent
    splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
        encoding_name='cl100k_base',
        chunk_size=512,
        chunk_overlap=20,
        disallowed_special=()
    )
    chunks = splitter.split_documents(all_docs)

    if not chunks:
        raise ValueError("No document chunks found. Check your PDFs or splitter.")

    print(f"Loaded {len(all_docs)} total pages from {len(pdf_paths)} papers, created {len(chunks)} chunks")
    return chunks

## Embedding Generation and Vector Storage

**Architecture mapping**: `Embeddings (Vector Storage)`

Embeddings convert text into numerical vectors that capture semantic meaning.
ChromaDB stores these vectors for fast similarity search.

**Model Caching concept**: Once embeddings are generated, they are stored in
ChromaDB and reused across all queries. This avoids recomputing embeddings
per query - a key cost and latency optimization in LLMOps.

In [ ]:
# ============================================================================
# TASK: Create Vector Store from Document Chunks
# ============================================================================
# This implements the "Embeddings (Vector Storage)" box in the architecture.
# cache_policy=None ensures fresh embeddings each run (for demo purposes).
# In production, you would persist the vector store to disk/cloud.
# ============================================================================

@task(cache_policy=None)
def create_vectorstore(chunks):
    """Create a ChromaDB vector database from document chunks.

    LLMOps concepts:
    - Embeddings/Vector Storage: ChromaDB stores embeddings for similarity search
    - Model Caching: Embeddings computed once and reused across all queries
    - Addresses the 'Inference Data Handling' challenge: integrates real-time
      external knowledge sources for contextual accuracy
    """
    # OpenAIEmbeddings converts text -> 1536-dimensional vectors
    embeddings = OpenAIEmbeddings(
        openai_api_key=OPENAI_API_KEY,
        openai_api_base=OPENAI_API_BASE
    )

    # Chroma.from_documents: embeds all chunks and stores them in-memory
    vectordb = Chroma.from_documents(chunks, embeddings)

    # Create a retriever interface for similarity search
    # k=5 means return the 5 most similar chunks for each query
    retriever = vectordb.as_retriever(
        search_type='similarity',
        search_kwargs={'k': 5}
    )

    print(f"Vector store created with {vectordb._collection.count()} embeddings")
    return retriever

## Chunk Analysis - LLMOps Feature Engineering

In traditional MLOps, you analyze feature distributions. In LLMOps, the equivalent
is analyzing your **chunk distribution** - are chunks the right size? Is information
evenly distributed? This affects retrieval quality.

In [ ]:
# ============================================================================
# TASK: Analyze Chunk Statistics
# ============================================================================
# This is the LLMOps equivalent of feature analysis in MLOps.
# Understanding your chunk distribution helps optimize retrieval quality.
# ============================================================================

@task
def analyze_chunks(chunks):
    """Analyze the token distribution of document chunks.

    LLMOps concept: Feature Engineering equivalent
    - In MLOps, you'd analyze feature distributions
    - In LLMOps, you analyze chunk size distributions
    - Uneven distributions may indicate poor chunking strategy
    """
    enc = tiktoken.get_encoding("cl100k_base")
    token_counts = [len(enc.encode(c.page_content)) for c in chunks]

    print(f"\nChunk Statistics (LLMOps Feature Analysis):")
    print(f"  Total chunks:     {len(chunks)}")
    print(f"  Avg tokens/chunk: {sum(token_counts)/len(token_counts):.0f}")
    print(f"  Min tokens:       {min(token_counts)}")
    print(f"  Max tokens:       {max(token_counts)}")
    print(f"  Total tokens:     {sum(token_counts)}")

    return token_counts

# Prompt Engineering and Versioning

**Architecture mapping**: `Fine-Tuning / Few-Shot Learning` + `Prompt Engineering`

This is one of the **key differentiators** between MLOps and LLMOps:
- In MLOps, you tune **hyperparameters** and retrain models
- In LLMOps, the **prompt IS the program** - changing it changes behavior entirely

### LLM Trade-offs Spectrum

```
  Prompt Engineering  -->  RAG, Agents  -->  Fine-Tuning  -->  Build from Scratch
  (least complex)                                              (most complex)
  (least data needed)                                          (most data needed)
```

Our agent uses **Prompt Engineering + RAG/Agent** - the sweet spot for most business use cases.
The detailed agent prompt (v2) provides **in-context learning** - a form of few-shot learning
where we teach the model through structured instructions in the prompt itself.

In [ ]:
# ============================================================================
# AGENT SYSTEM PROMPTS - Five Versions for A/B Testing
# ============================================================================
# Prompt versioning is the LLMOps equivalent of model versioning in MLOps.
# We create five versions to measure how prompt design affects agent behavior.
#
# v1: Simple, minimal instructions
# v2: Detailed workflow with structured output rules
# v3: Role-playing expert with citation requirements
# v4: Chain-of-thought reasoning with step-by-step analysis
# v5: Comparative analysis specialist
#
# All are logged to MLflow for comparison.
# ============================================================================

# --- Version 1: Simple Agent Prompt ---
agent_system_prompt_v1 = """You are an assistant that answers questions about LLM security 
and safety research papers covering topics like risk assessment, prompt injection, 
AI safety frameworks, and content moderation guardrails.

You have access to tools for searching the documents, summarizing sections, and 
performing analysis. Use the appropriate tool for each question.
Answer based only on information found in the documents."""

# --- Version 2: Detailed Workflow ---
agent_system_prompt_v2 = """You are an expert AI security researcher analyzing a corpus of 
research papers on LLM security and safety, including risk assessment methodologies, 
prompt injection attacks, AI safety taxonomies, and content moderation guardrails.

You have access to specialized tools. Follow this workflow:
1. ALWAYS use document_search first to find relevant passages
2. If the question asks for a summary, also use summarize_section
3. If the question requires analysis or comparison, use reasoning after gathering context
4. Synthesize information from all tool outputs into a coherent answer

Rules:
- Only answer based on information found in the documents
- If the documents don't contain relevant information, say so explicitly
- Cite specific details and paper names when possible
- Structure complex answers with bullet points"""

# --- Version 3: Citation-Heavy Academic Expert ---
agent_system_prompt_v3 = """You are a peer reviewer for a top-tier AI security conference. 
You are analyzing a corpus of research papers on LLM security and safety.

For every claim you make, you MUST:
1. Use document_search to find the specific passage that supports it
2. Cite the paper by author name and include the page number
3. Use direct quotes from the papers when possible

Format your answers as an academic review:
- Start with a one-sentence summary of your finding
- Support each point with evidence from the papers
- Note any contradictions or gaps between papers
- End with a brief assessment of the evidence quality

Only answer based on information found in the documents."""

# --- Version 4: Chain-of-Thought Reasoner ---
agent_system_prompt_v4 = """You are a methodical AI safety analyst who thinks step by step.

When answering questions about the LLM security research papers:

Step 1: SEARCH - Use document_search to gather all relevant passages
Step 2: IDENTIFY - List the key facts found in each retrieved passage
Step 3: REASON - Use the reasoning tool to analyze relationships between facts
Step 4: SYNTHESIZE - Combine findings into a clear, logical answer

Show your reasoning process explicitly. For each step, explain:
- What you searched for and why
- What you found
- How it connects to the question

Only answer based on information found in the documents.
If information is insufficient, state what is missing."""

# --- Version 5: Comparative Analysis Specialist ---
agent_system_prompt_v5 = """You are a comparative analysis specialist focusing on LLM security 
research. Your expertise is finding connections, contradictions, and complementary 
findings across multiple papers.

When answering questions:
1. Use document_search multiple times with different queries to gather perspectives 
   from different papers
2. Use summarize_section to distill key findings from each paper
3. Use reasoning to compare and contrast findings across papers

Structure your answers as a comparison:
- What each paper says about the topic
- Where papers agree or complement each other
- Where papers disagree or present different perspectives
- What the combined evidence suggests

Always identify which paper each finding comes from.
Only answer based on information found in the documents."""

# Dictionary for version management
AGENT_PROMPT_VERSIONS = {
    "v1": agent_system_prompt_v1,
    "v2": agent_system_prompt_v2,
    "v3": agent_system_prompt_v3,
    "v4": agent_system_prompt_v4,
    "v5": agent_system_prompt_v5,
}

print(f"Defined {len(AGENT_PROMPT_VERSIONS)} prompt versions for A/B testing")

In [ ]:
# ============================================================================
# PROMPT TEMPLATE BUILDER
# ============================================================================
# The ChatPromptTemplate structures how the agent receives instructions.
# The 'agent_scratchpad' placeholder is where LangChain records the agent's
# reasoning steps - tool calls and observations. MLflow autolog captures
# all of this automatically as trace spans.
# ============================================================================

def build_agent_prompt(system_prompt):
    """Build a ChatPromptTemplate for the agent.

    Components:
    - system: The agent's personality and instructions
    - human {input}: The user's question
    - agent_scratchpad: LangChain fills this with the agent's reasoning
      (tool calls, observations, intermediate thoughts)
    """
    return ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", "{input}"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ])


def log_prompt_version(version_name, prompt_text):
    """Log a prompt version to MLflow for traceability.

    LLMOps concept: Prompt Versioning
    - Just as MLOps versions model weights, LLMOps versions prompts
    - Each prompt version is stored as an artifact in MLflow
    - Allows comparing which prompt version produces better results
    """
    mlflow.log_text(prompt_text, f"prompts/{version_name}_system_prompt.txt")
    mlflow.log_param("prompt_version", version_name)
    mlflow.log_param("prompt_length_chars", len(prompt_text))

# Building the Intelligent Agent

**Architecture mapping**: `Pre-Trained LLM -> Context-Specific LLM -> LLM API`

This is the **centerpiece** of the notebook. Instead of a fixed RAG chain
(retrieve -> generate), we build an **agent** that can reason about which tool to use.

### Why Agents Over RAG Chains?

| RAG Chain (Old Approach) | Agent (Our Approach) |
|---|---|
| Always follows: retrieve -> generate | Decides which tool to use per question |
| Can't search multiple times | Can search with different queries |
| No summarization capability | Can summarize before answering |
| No analytical reasoning | Can reason about retrieved content |
| Fixed pipeline | Flexible, adaptive workflow |

The agent transforms a **Pre-Trained LLM** into a **Context-Specific LLM** by giving
it domain-specific tools and instructions. This is exactly what the architecture
diagram shows.

## Tool 1: Document Search (RAG Retrieval)

This tool wraps the vector database retrieval. When the agent calls it,
it searches ChromaDB for the most relevant chunks. This is the RAG component
from the architecture diagram.

In [ ]:
# ============================================================================
# TOOL 1: DOCUMENT SEARCH (RAG RETRIEVAL)
# ============================================================================
# This is the core RAG tool. It searches the vector store for passages
# that are semantically similar to the query.
#
# MLflow autolog automatically traces each invocation:
#   - Input query
#   - Retrieved documents
#   - Latency
# ============================================================================

def create_document_search_tool(retriever):
    """Create a RAG-based document search tool for the agent."""

    def search_documents(query: str) -> str:
        """Search the LLM security research papers for passages relevant to the query."""
        docs = retriever.invoke(query)
        if not docs:
            return "No relevant documents found."
        # Format results with source page numbers for citation
        results = []
        for i, doc in enumerate(docs):
            page = doc.metadata.get('page', 'N/A')
            results.append(f"[Source {i+1}] (Page {page}):\n{doc.page_content}")
        return "\n\n".join(results)

    return Tool(
        name="document_search",
        description=(
            "Search the LLM security and safety research papers for specific information. "
            "Use this tool to find relevant passages about risk assessment, "
            "prompt injection, AI safety frameworks, content moderation, or any security topic."
        ),
        func=search_documents
    )

## Tool 2: Document Summarizer

This tool first retrieves relevant content, then asks the LLM to summarize it.
It chains two operations: retrieval + generation. MLflow traces both as nested spans.

In [ ]:
# ============================================================================
# TOOL 2: DOCUMENT SUMMARIZER
# ============================================================================
# This tool chains two operations:
#   1. Retrieve relevant passages from the vector store
#   2. Ask the LLM to summarize them into bullet points
#
# MLflow traces both operations as nested spans within the tool call:
#   AgentExecutor -> summarize_section -> [retriever call] + [LLM call]
# ============================================================================

def create_summarizer_tool(retriever, llm_instance):
    """Create a summarization tool that retrieves then summarizes."""

    def summarize_topic(topic: str) -> str:
        """Summarize a specific topic from the LLM security research papers."""
        # Step 1: Retrieve relevant passages
        docs = retriever.invoke(topic)
        if not docs:
            return "No relevant content found to summarize."
        context = "\n".join([d.page_content for d in docs])

        # Step 2: Generate a concise summary using the LLM
        response = llm_instance.invoke(
            f"Summarize the following content about '{topic}' in 3-4 concise "
            f"bullet points:\n\n{context}"
        )
        return response.content

    return Tool(
        name="summarize_section",
        description=(
            "Summarize a specific topic or section from the LLM security research papers. "
            "Use when asked for overviews, summaries, or key takeaways."
        ),
        func=summarize_topic
    )

## Tool 3: Reasoning and Analysis

This tool handles analytical questions that go beyond simple retrieval.
It demonstrates the **Context-Specific LLM** concept - using the LLM as
a reasoning engine for complex business analysis.

In [ ]:
# ============================================================================
# TOOL 3: REASONING AND ANALYSIS
# ============================================================================
# This tool handles analytical questions. The agent can call it after
# gathering facts via document_search to perform deeper analysis.
#
# LLMOps concept: The LLM as a reasoning engine within the agent architecture.
# ============================================================================

def create_reasoning_tool(llm_instance):
    """Create a reasoning/analysis tool for complex questions."""

    def analyze(query: str) -> str:
        """Perform analytical reasoning about LLM security and safety concepts."""
        response = llm_instance.invoke(
            f"Analyze the following LLM security question. Provide structured "
            f"reasoning with clear logic:\n\n{query}"
        )
        return response.content

    return Tool(
        name="reasoning",
        description=(
            "Perform analytical reasoning about business concepts from the "
            "Apple report. Use for comparative analysis, implications, or "
            "strategic insights after gathering facts."
        ),
        func=analyze
    )

## Assembling the Agent

We combine the LLM, tools, and prompt into an `AgentExecutor`. This is where
the **Pre-Trained LLM** becomes a **Context-Specific LLM** - the agent can now
search documents, summarize, and reason.

### Inferencing Challenges Addressed

| Challenge (from slides) | How We Address It |
|---|---|
| **Speed & Latency** | `max_iterations=5` limits runaway agent loops |
| **Cost Efficiency** | Agent skips unnecessary tool calls |
| **Security Risks** | `handle_parsing_errors=True` prevents prompt injection crashes |
| **Scalability** | Agent reasoning is logged via MLflow for optimization |

In [ ]:
# ============================================================================
# TASK: Build the Agent with Tools
# ============================================================================
# This assembles all components into a working agent:
#   Pre-Trained LLM + Tools + Prompt = Context-Specific LLM (Agent)
#
# create_openai_tools_agent: Uses OpenAI's function-calling API to let the
#   model decide which tool to invoke. The model outputs a structured tool
#   call (function name + arguments) rather than free text.
#
# AgentExecutor: Runs the agent in a loop:
#   1. Agent decides which tool to call
#   2. Tool executes and returns observation
#   3. Agent decides next action (call another tool or return answer)
#   4. Repeat until done or max_iterations reached
# ============================================================================

@task(cache_policy=None)
def build_agent(retriever, prompt_version="v1"):
    """Build a LangChain agent with document search, summarization, and reasoning tools."""
    # Get the system prompt for this version
    system_prompt = AGENT_PROMPT_VERSIONS[prompt_version]
    prompt = build_agent_prompt(system_prompt)

    # Create the three tools
    search_tool = create_document_search_tool(retriever)
    summarizer_tool = create_summarizer_tool(retriever, llm)
    reasoning_tool = create_reasoning_tool(llm)
    tools = [search_tool, summarizer_tool, reasoning_tool]

    # Create the agent using OpenAI's function-calling mechanism
    agent = create_openai_tools_agent(llm, tools, prompt)

    # Wrap in executor with safety controls
    agent_executor = AgentExecutor(
        agent=agent,
        tools=tools,
        verbose=True,                    # Print reasoning steps to console
        max_iterations=5,                # Cost control: limit agent loops
        return_intermediate_steps=True,  # Capture tool calls for analysis
        handle_parsing_errors=True       # Security: handle malformed outputs
    )

    print(f"Agent built with prompt {prompt_version} and {len(tools)} tools: "
          f"{[t.name for t in tools]}")
    return agent_executor, tools

# Agent Pipeline with MLflow Tracing

**Architecture mapping**: Full end-to-end flow covering
**Model Versioning** + **Model Monitoring**

Each agent run is:
- **Orchestrated** by Prefect (`@flow` / `@task` decorators)
- **Traced** by MLflow (every tool call, LLM invocation, latency, tokens)
- **Versioned** (prompt version logged as a parameter)
- **Evaluated** (RAGAS metrics for faithfulness and relevancy)

In [ ]:
# ============================================================================
# TASK: Execute an Agent Query
# ============================================================================
# Runs the agent on a single question and captures:
#   - The final answer
#   - All intermediate steps (tool calls with inputs/outputs)
#   - Execution time
#   - Retrieved contexts (for evaluation)
#   - Token usage and estimated cost
#
# MLflow autolog automatically captures the full trace hierarchy.
# ============================================================================

# gpt-5-mini pricing (per 1M tokens)
GPT5_MINI_INPUT_COST = 0.30    # $0.30 per 1M input tokens
GPT5_MINI_OUTPUT_COST = 1.25   # $1.25 per 1M output tokens

@task(cache_policy=None)
def run_agent_query(agent_executor, question):
    """Execute an agent query and capture detailed execution information."""
    start_time = time.time()

    # Use get_openai_callback to capture token usage across all LLM calls
    with get_openai_callback() as cb:
        result = agent_executor.invoke({"input": question})

    elapsed = time.time() - start_time

    answer = result["output"]
    steps = result.get("intermediate_steps", [])

    # Extract tool usage information for logging
    tool_calls = []
    contexts = []
    for step in steps:
        action, observation = step
        tool_calls.append({
            "tool": action.tool,
            "input": str(action.tool_input),
            "output": str(observation)[:500]  # Truncate for logging
        })
        # Collect contexts from document_search for RAGAS evaluation
        if action.tool == "document_search":
            contexts.append(str(observation))

    # Calculate cost
    input_cost = (cb.prompt_tokens / 1_000_000) * GPT5_MINI_INPUT_COST
    output_cost = (cb.completion_tokens / 1_000_000) * GPT5_MINI_OUTPUT_COST
    total_cost = input_cost + output_cost

    token_usage = {
        "prompt_tokens": cb.prompt_tokens,
        "completion_tokens": cb.completion_tokens,
        "total_tokens": cb.total_tokens,
        "input_cost_usd": input_cost,
        "output_cost_usd": output_cost,
        "total_cost_usd": total_cost,
    }

    return answer, contexts, tool_calls, elapsed, token_usage

In [ ]:
# ============================================================================
# TASK: Evaluate Agent Output with RAGAS
# ============================================================================
# RAGAS (Retrieval Augmented Generation Assessment) measures:
#   - Faithfulness: Is the answer grounded in retrieved context?
#   - Answer Relevancy: Does the answer address the question?
#
# LLMOps concept: Evaluation is shared between MLOps and LLMOps (Venn diagram).
# For agents, we evaluate not just the final answer but the quality of retrieval.
# ============================================================================

@task
def evaluate_agent_output(question, answer, contexts):
    """Evaluate agent output using RAGAS metrics."""
    if not contexts:
        print("  No contexts available for RAGAS evaluation, skipping.")
        return {"faithfulness": None, "answer_relevancy": None}

    dataset = Dataset.from_dict({
        "question": [question],
        "answer": [answer],
        "contexts": [contexts]
    })

    result = ragas_evaluate(
        dataset=dataset,
        metrics=[Faithfulness(), AnswerRelevancy()]
    )
    return result

In [ ]:
# ============================================================================
# FLOW: Complete Agent Pipeline
# ============================================================================
# This Prefect @flow orchestrates the entire LLMOps lifecycle:
#
#   1. Data Processing Pipeline    -> load_and_split_pdfs()
#   2. Embeddings / Vector Storage  -> create_vectorstore()
#   3. Feature Analysis             -> analyze_chunks()
#   4. Context-Specific LLM (Agent) -> build_agent()
#   5. LLM API / End User Query     -> run_agent_query()
#   6. Model Monitoring             -> MLflow metrics + traces
#   7. Model Versioning             -> Prompt versions logged to MLflow
#   8. Evaluation                   -> evaluate_agent_output()
#
# Each step maps to a component in the LLMOps architecture diagram.
# ============================================================================

@flow(name="agent_llmops_pipeline")
def agent_pipeline(pdf_paths, question, versions=["v1", "v2", "v3", "v4", "v5"]):
    """Full LLMOps pipeline: data processing -> agent execution -> evaluation."""

    # ---- Step 1: Data Processing Pipeline ----
    chunks = load_and_split_pdfs(pdf_paths)
    token_stats = analyze_chunks(chunks)

    # ---- Step 2: Create Embeddings / Vector Storage ----
    retriever = create_vectorstore(chunks)

    all_results = []

    # ---- Step 3: Test Multiple Prompt Versions ----
    # This is the Prompt Versioning component of LLMOps.
    # We run the same question with different prompts to compare results.
    for prompt_version in versions:

        # ---- Step 4: Build Context-Specific Agent ----
        agent_executor, tools = build_agent(retriever, prompt_version)

        # ---- Step 5: Execute with MLflow Tracking ----
        with mlflow.start_run(run_name=f"Agent_{prompt_version}", nested=True):

            # Log versioning information (Model Versioning concept)
            log_prompt_version(prompt_version, AGENT_PROMPT_VERSIONS[prompt_version])
            mlflow.log_param("question", question[:250])  # Truncate for MLflow
            mlflow.log_param("num_tools", len(tools))
            mlflow.log_param("tool_names", str([t.name for t in tools]))
            mlflow.log_param("model_name", "gpt-5-mini")
            mlflow.log_param("temperature", 1.0)

            # Run the agent
            answer, contexts, tool_calls, elapsed, token_usage = run_agent_query(
                agent_executor, question
            )

            # Log monitoring metrics (Model Monitoring concept)
            mlflow.log_metric("response_time_seconds", elapsed)
            mlflow.log_metric("num_tool_calls", len(tool_calls))
            mlflow.log_metric("answer_length", len(answer))
            mlflow.log_metric("num_contexts_retrieved", len(contexts))

            # Log token usage and cost metrics
            mlflow.log_metric("prompt_tokens", token_usage["prompt_tokens"])
            mlflow.log_metric("completion_tokens", token_usage["completion_tokens"])
            mlflow.log_metric("total_tokens", token_usage["total_tokens"])
            mlflow.log_metric("input_cost_usd", token_usage["input_cost_usd"])
            mlflow.log_metric("output_cost_usd", token_usage["output_cost_usd"])
            mlflow.log_metric("total_cost_usd", token_usage["total_cost_usd"])

            # Log artifacts (answer text and tool call details)
            mlflow.log_text(answer, f"answer_{prompt_version}.txt")
            mlflow.log_text(
                json.dumps(tool_calls, indent=2),
                f"tool_calls_{prompt_version}.json"
            )

            # ---- Step 6: Evaluate ----
            eval_result = evaluate_agent_output(question, answer, contexts)

            # Log evaluation metrics to MLflow
            try:
                df = eval_result.to_pandas()
                for col in df.columns:
                    val = df[col].iloc[0]
                    if isinstance(val, (int, float)):
                        mlflow.log_metric(str(col), float(val))
                print(f"  Evaluation metrics logged for {prompt_version}")
            except Exception as e:
                print(f"  Could not log eval metrics for {prompt_version}: {e}")

            # Collect results for comparison
            all_results.append({
                "version": prompt_version,
                "answer": answer,
                "tool_calls": tool_calls,
                "elapsed": elapsed,
                "token_usage": token_usage,
                "eval": eval_result
            })

            print(f"\n{'='*60}")
            print(f"Agent {prompt_version} Results:")
            print(f"  Answer: {answer[:200]}...")
            print(f"  Tools used: {[tc['tool'] for tc in tool_calls]}")
            print(f"  Response time: {elapsed:.2f}s")
            print(f"  Tokens: {token_usage['prompt_tokens']} in / {token_usage['completion_tokens']} out")
            print(f"  Cost: ${token_usage['total_cost_usd']:.6f}")
            print(f"{'='*60}\n")

    return all_results

## Start Prefect Server

Prefect provides a web UI for monitoring flow runs, task states, and pipeline health.
We start the server **before** running the pipeline so that all flow runs are tracked.
The UI will be exposed via Cloudflare Tunnel later alongside the MLflow UI.

In [ ]:
# ============================================================================
# START PREFECT SERVER
# ============================================================================
# Prefect's UI lets you visually explore:
#   - Flow runs and their states (completed, failed, pending)
#   - Task-level execution details and timing
#   - Pipeline DAG visualization
#   - Logs from each task
#
# We start the server here so flows report to it during execution.
# The UI will be exposed via Cloudflare Tunnel after the MLflow UI.
# ============================================================================

prefect_port = 4200

# Start Prefect server as a background process
prefect_process = subprocess.Popen(
    f"prefect server start --host 0.0.0.0 --port {prefect_port}",
    shell=True,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# Wait for the server to start
time.sleep(5)

# Set the Prefect API URL so flows report to this server
os.environ["PREFECT_API_URL"] = f"http://127.0.0.1:{prefect_port}/api"

print(f"Prefect server started on port {prefect_port}")
print(f"PREFECT_API_URL set to http://127.0.0.1:{prefect_port}/api")
print("Pipeline flow runs will now be tracked by Prefect.")

# Question Answering with the Agent

**Architecture mapping**: `End User Apps` (Users -> Prompts, Tasks/Queries)

We now run the agent pipeline with real questions. Each question demonstrates
a different capability:
1. **Factual retrieval** - uses `document_search`
2. **Summarization** - uses `document_search` + `summarize_section`
3. **Analytical reasoning** - uses `document_search` + `reasoning`

Both prompt versions (v1 and v2) are tested for each question.

### Question 1: Factual Retrieval

This question requires finding specific information across the research papers.
The agent should use `document_search` to retrieve relevant passages about
LLM threat categorization.

In [ ]:
question_1 = "What are the OWASP Top 10 risks for LLM applications and how does the threat matrix map them to different stakeholder groups?"
results_1 = agent_pipeline(pdf_files, question_1)

### Question 2: Summarization Task

This question asks for a structured summary. The v2 prompt (with explicit workflow
instructions) should trigger the `summarize_section` tool, while v1 might just
use `document_search`. This demonstrates how **prompt engineering** affects agent behavior.

In [ ]:
question_2 = "Summarize the Llama Guard safety risk taxonomy categories in bulleted points and explain each category in under two lines."
results_2 = agent_pipeline(pdf_files, question_2)

### Question 3: Analytical Reasoning

This question requires cross-paper analysis and deeper reasoning.
The agent should combine `document_search` with `reasoning` to synthesize
insights from multiple papers.

In [ ]:
question_3 = "How do indirect prompt injection attacks described by Greshake et al. relate to the three pillars of AI safety (Trustworthy, Responsible, Safe AI) from Chen et al.'s framework?"
results_3 = agent_pipeline(pdf_files, question_3)

### Comparing Agent Behavior Across Prompt Versions

By running the same questions with different prompts, we can measure how
prompt engineering affects agent behavior. This is a key LLMOps practice:
systematic evaluation of prompt changes.

**Semantic Drift Awareness**: In production, you would monitor these metrics
over time. A sudden drop in relevancy or change in tool usage patterns could
indicate **semantic drift** - the LLMOps equivalent of data drift in MLOps.

In [ ]:
# ============================================================================
# PROMPT VERSION COMPARISON SUMMARY
# ============================================================================
# Compare all five prompt versions side by side.
# ============================================================================

print("=" * 80)
print("PROMPT VERSION COMPARISON SUMMARY")
print("=" * 80)

# Use the most recent results (from the last question)
latest_results = results_3  # Cross-paper analysis question

total_pipeline_cost = 0.0

for r in latest_results:
    v = r["version"]
    tu = r["token_usage"]
    total_pipeline_cost += tu["total_cost_usd"]
    print(f"\n--- {v} ---")
    print(f"  Response time: {r['elapsed']:.2f}s")
    print(f"  Tools used:    {[tc['tool'] for tc in r['tool_calls']]}")
    print(f"  Answer length: {len(r['answer'])} chars")
    print(f"  Tokens:        {tu['prompt_tokens']} in / {tu['completion_tokens']} out / {tu['total_tokens']} total")
    print(f"  Cost:          ${tu['total_cost_usd']:.6f} (input: ${tu['input_cost_usd']:.6f} + output: ${tu['output_cost_usd']:.6f})")
    try:
        df = r["eval"].to_pandas()
        scores = {col: f"{df[col].iloc[0]:.3f}" for col in df.columns if isinstance(df[col].iloc[0], (int, float))}
        print(f"  RAGAS scores:  {scores}")
    except Exception:
        pass

print(f"\n{'='*80}")
print(f"Total cost for this question (all {len(latest_results)} versions): ${total_pipeline_cost:.6f}")
print(f"{'='*80}")

# Model Monitoring with MLflow Tracing

**Architecture mapping**: `Model Monitoring`

Monitoring is a shared concern between MLOps and LLMOps. For agents, monitoring
is more complex because we need to track:
- **Intermediate reasoning steps** (which tools were called and why)
- **Tool selection patterns** (is the agent using the right tools?)
- **Per-tool latency** (which tools are slow?)
- **Token usage** (what's the cost per query?)

### What MLflow Autolog Captures Automatically

| What | How |
|---|---|
| **Traces** | Hierarchical view: AgentExecutor -> tool call -> LLM call |
| **Spans** | Individual steps with start/end time, status, inputs, outputs |
| **Token usage** | Input and output tokens per LLM call |
| **Latency** | Time per span and total execution time |
| **Tool calls** | Which tools were called, with what inputs, producing what outputs |

## Exposing MLflow and Prefect UIs in Colab

Both MLflow and Prefect provide web UIs for inspecting pipeline runs.
Since Colab runs on a remote VM, we use **Cloudflare Tunnel** (`cloudflared`) to create
a free public URL for the MLflow UI — no account or auth token required.

The Prefect UI is served directly via Colab's `serve_kernel_port_as_window`.

In [ ]:
# ============================================================================
# START MLFLOW SERVER
# ============================================================================
# We use 'mlflow server' with --allowed-hosts '*' so it accepts requests
# forwarded through the Cloudflare tunnel (which changes the Host header).
# ============================================================================

mlflow_uri = "file:////content/llmops_mlflow"
mlflow_port = 5000

# Start MLflow server as a background process
mlflow_process = subprocess.Popen(
    f"mlflow server --backend-store-uri {mlflow_uri} --host 0.0.0.0 --port {mlflow_port} "
    f"--allowed-hosts '*' --cors-allowed-origins '*'",
    shell=True,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# Wait for server to fully start (gunicorn needs time to initialize workers)
print(f"Starting MLflow server on port {mlflow_port}...")
for attempt in range(15):
    time.sleep(2)
    try:
        import urllib.request
        urllib.request.urlopen(f"http://127.0.0.1:{mlflow_port}/")
        print(f"MLflow server ready (took {(attempt+1)*2}s)")
        break
    except Exception:
        pass
else:
    print("Warning: MLflow server may still be starting up")

In [ ]:
# ============================================================================
# CREATE CLOUDFLARE TUNNEL FOR MLFLOW UI
# ============================================================================
# cloudflared creates a public URL that tunnels to our local MLflow server
# using Cloudflare's free trycloudflare.com service (no account needed).
# ============================================================================

import platform
import re
import shutil
import stat
import subprocess
import time
import urllib.request
from pathlib import Path

cloudflared_path = shutil.which("cloudflared")
cloudflared_log = Path("/tmp/cloudflared.log")

if cloudflared_path is None:
    cloudflared_path = Path("/tmp/cloudflared")

    def _cloudflared_works(binary_path: Path) -> bool:
        if not binary_path.exists():
            return False
        binary_path.chmod(binary_path.stat().st_mode | stat.S_IEXEC)
        result = subprocess.run(
            [str(binary_path), "--version"],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )
        return result.returncode == 0

    if not _cloudflared_works(cloudflared_path):
        machine = platform.machine().lower()
        if machine in {"x86_64", "amd64"}:
            cloudflared_asset = "cloudflared-linux-amd64"
        elif machine in {"aarch64", "arm64"}:
            cloudflared_asset = "cloudflared-linux-arm64"
        else:
            raise RuntimeError(f"Unsupported architecture for cloudflared in Colab: {machine}")

        cloudflared_url = (
            "https://github.com/cloudflare/cloudflared/releases/latest/download/"
            f"{cloudflared_asset}"
        )

        last_error = None
        for attempt in range(3):
            try:
                request = urllib.request.Request(
                    cloudflared_url,
                    headers={"User-Agent": "Mozilla/5.0"},
                )
                with urllib.request.urlopen(request) as response, open(cloudflared_path, "wb") as binary_file:
                    binary_file.write(response.read())
                cloudflared_path.chmod(cloudflared_path.stat().st_mode | stat.S_IEXEC)
                if not _cloudflared_works(cloudflared_path):
                    raise RuntimeError("Downloaded cloudflared binary did not start correctly")
                break
            except Exception as exc:
                last_error = exc
                cloudflared_path.unlink(missing_ok=True)
                time.sleep(2 ** attempt)
        else:
            raise RuntimeError("Failed to install cloudflared from GitHub releases") from last_error
else:
    cloudflared_path = Path(cloudflared_path)

# Stop any existing cloudflared tunnel from a previous run of this cell.
try:
    if "cloudflared_process" in globals() and cloudflared_process.poll() is None:
        cloudflared_process.terminate()
        cloudflared_process.wait(timeout=5)
except Exception:
    pass

cloudflared_log.write_text("")

cloudflared_process = subprocess.Popen(
    [
        str(cloudflared_path),
        "tunnel",
        "--url",
        f"http://127.0.0.1:{mlflow_port}",
        "--no-autoupdate",
        "--logfile",
        str(cloudflared_log),
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

mlflow_url = None
url_pattern = re.compile(r"https://[A-Za-z0-9-]+\.trycloudflare\.com")

for _ in range(90):
    if cloudflared_process.poll() is not None:
        logs = cloudflared_log.read_text(errors="ignore")
        raise RuntimeError(
            "cloudflared exited before creating a tunnel.\n\n"
            f"Recent cloudflared logs:\n{logs[-2000:]}"
        )

    logs = cloudflared_log.read_text(errors="ignore")
    match = url_pattern.search(logs)
    if match:
        mlflow_url = match.group(0)
        break

    time.sleep(1)

if not mlflow_url:
    logs = cloudflared_log.read_text(errors="ignore")
    raise TimeoutError(
        "Timed out waiting for cloudflared to create a public URL.\n\n"
        f"Recent cloudflared logs:\n{logs[-2000:]}"
    )

print(f"\n{'='*60}")
print(f"  MLflow UI:  {mlflow_url}")
print(f"{'='*60}")
print(f"\nCloudflare Tunnel is forwarding to http://127.0.0.1:{mlflow_port}")
print(f"Tunnel log: {cloudflared_log}")
print(f"\nIn MLflow, navigate to:")
print(f"  1. Click 'LLMOps_Agent_Demo' experiment")
print(f"  2. Compare runs for v1 vs v2 prompt versions")
print(f"  3. Click into a run -> 'Traces' tab to see the agent execution tree")
print(f"  4. Examine tool calls, latency per step, and token counts")

# Prefect UI is available locally on the Colab VM
print(f"\nPrefect UI is running on port {prefect_port}")
print(f"Use Colab's port forwarding or the cell below to access it.")


## Programmatic Trace Analysis

Beyond the UI, MLflow lets you query experiment data programmatically.
This is essential for **automated monitoring** in production - you can
set up alerts for response time spikes, evaluation drops, or unusual
tool usage patterns.

In [ ]:
# ============================================================================
# PROGRAMMATIC TRACE ANALYSIS
# ============================================================================
# In production, you would run queries like these on a schedule to detect:
#   - Response time degradation (latency alerts)
#   - Evaluation score drops (quality alerts)
#   - Unusual tool usage patterns (anomaly detection)
# ============================================================================

experiment = mlflow.get_experiment_by_name("LLMOps_Agent_Demo")
runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

# Display key metrics for all tracked runs
display_cols = [
    "run_id", "params.prompt_version",
    "metrics.response_time_seconds", "metrics.num_tool_calls",
    "metrics.answer_length"
]
# Only show columns that exist in the data
available_cols = [c for c in display_cols if c in runs.columns]

print("\nAll Tracked Runs:")
print(runs[available_cols].to_string())

# Human Feedback in LLMOps (RLHF Pattern)

**Architecture mapping**: `Reinforcement Learning from Human Feedback (RLHF)`

The LLMOps architecture diagram includes RLHF as a feedback loop. While full RLHF
requires model retraining, the practical LLMOps approach includes:

1. **Collect** human ratings on agent outputs
2. **Log** ratings to MLflow alongside the agent's run data
3. **Analyze** which prompt versions receive better ratings
4. **Improve** prompts based on feedback patterns

This is "Human Feedback" from the MLOps vs LLMOps Venn diagram - a capability
unique to LLMOps that doesn't exist in traditional ML pipelines.

In [ ]:
# ============================================================================
# HUMAN FEEDBACK COLLECTION PATTERN
# ============================================================================
# In production, this would be a UI where reviewers rate agent responses.
# Here we simulate feedback to demonstrate the logging pattern.
#
# The feedback loop:
#   Agent answer -> Human rates it -> Logged to MLflow -> Prompt improved
#
# This is how the RLHF box in the architecture connects back to the agent:
# human feedback drives prompt engineering iterations.
# ============================================================================

def collect_human_feedback(question, answer, prompt_version):
    """Simulate collecting and logging human feedback on agent responses.

    In production, replace the random scores with actual human ratings
    from a review interface (e.g., thumbs up/down, 1-5 scale).
    """
    # Simulated feedback (in production, this comes from a reviewer UI)
    feedback = {
        "relevance": round(random.uniform(3.5, 5.0), 2),
        "accuracy": round(random.uniform(3.5, 5.0), 2),
        "helpfulness": round(random.uniform(3.5, 5.0), 2),
        "would_improve": "More specific citations from the document would help"
    }

    with mlflow.start_run(run_name=f"Feedback_{prompt_version}", nested=True):
        mlflow.log_param("feedback_type", "human_review")
        mlflow.log_param("prompt_version", prompt_version)
        mlflow.log_param("question", question[:250])
        mlflow.log_metrics({
            "human_relevance": feedback["relevance"],
            "human_accuracy": feedback["accuracy"],
            "human_helpfulness": feedback["helpfulness"]
        })
        mlflow.log_text(
            json.dumps(feedback, indent=2),
            f"human_feedback_{prompt_version}.json"
        )

    print(f"  Feedback for {prompt_version}: relevance={feedback['relevance']}, "
          f"accuracy={feedback['accuracy']}, helpfulness={feedback['helpfulness']}")
    return feedback


# Collect feedback for both prompt versions on question 1
print("Collecting human feedback (simulated):")
for version in ["v1", "v2"]:
    idx = 0 if version == "v1" else 1
    collect_human_feedback(
        question_1,
        results_1[idx]["answer"],
        version
    )

print("\nIn production, these ratings would drive prompt improvement iterations.")
print("The version with higher human ratings would be promoted as the default.")

## Security Considerations

The research papers in our corpus directly address the security challenges of LLM deployment:

| Risk (from Papers) | Paper Source | LLMOps Mitigation in This Notebook |
|---|---|---|
| **Prompt injection** (direct & indirect) | Greshake et al., Pankajakshan et al. | `handle_parsing_errors=True` prevents crashes from malformed inputs |
| **Insecure output handling** | Pankajakshan et al. (OWASP LLM02) | Agent outputs are structured and tool-bounded |
| **Training data poisoning** | Pankajakshan et al. (OWASP LLM03) | Embeddings derived from vetted research papers only |
| **Runaway agent loops** | General LLMOps concern | `max_iterations=5` limits compute cost |
| **Content safety violations** | Inan et al. (Llama Guard taxonomy) | MLflow tracing detects anomalous outputs |
| **Lack of trustworthiness** | Chen et al. (AI Safety framework) | RAGAS evaluation + human feedback loop |

# Model Versioning and Deployment Readiness

**Architecture mapping**: `Model Versioning`

In LLMOps, **Model Versioning** goes beyond weight checkpoints. It includes
versioning the entire agent configuration:
- The **prompt** (which instructions the agent follows)
- The **tools** (what capabilities the agent has)
- The **LLM parameters** (model, temperature, max tokens)
- The **evaluation metrics** (how well the agent performs)

MLflow enables this through experiment runs and artifact logging.

In [ ]:
# ============================================================================
# LOG BEST AGENT CONFIGURATION AS A VERSIONED ARTIFACT
# ============================================================================
# After comparing prompt versions and reviewing evaluation metrics,
# we select the best configuration and log it as a deployment artifact.
#
# This is the "Model Versioning" component of LLMOps:
#   - The complete agent config is saved as a JSON artifact
#   - Tagged as deployment-ready for production promotion
#   - Can be loaded later to recreate the exact same agent
# ============================================================================

with mlflow.start_run(run_name="Agent_Best_Version"):
    # Select the best version (in practice, based on evaluation metrics)
    # Select best version based on results (default to v3 - citation-heavy)
    best_version = "v3"

    # Log the complete agent configuration
    agent_config = {
        "prompt_version": best_version,
        "system_prompt": AGENT_PROMPT_VERSIONS[best_version],
        "tools": ["document_search", "summarize_section", "reasoning"],
        "llm_model": "gpt-5-mini",
        "temperature": 1.0,
        "max_iterations": 5,
        "chunk_size": 512,
        "chunk_overlap": 20,
        "retriever_k": 5
    }

    mlflow.log_param("selected_version", best_version)
    mlflow.log_param("deployment_status", "ready")
    mlflow.log_text(
        json.dumps(agent_config, indent=2),
        "agent_config.json"
    )
    mlflow.set_tag("deployment_ready", "true")
    mlflow.set_tag("agent_type", "openai_tools_agent")

    print("Agent configuration logged as versioned artifact in MLflow")
    print(f"Selected version: {best_version}")
    print(f"Config: {json.dumps(agent_config, indent=2)}")

## Complete LLMOps Architecture Mapping

Here is how every component from the LLMOps architecture diagram was implemented:

| Architecture Component | Notebook Section | Implementation |
|---|---|---|
| **Data Processing Pipeline** | Section 3 | `PyMuPDFLoader` (multi-PDF) + `RecursiveCharacterTextSplitter` |
| **Embeddings / Vector Storage** | Section 3 | `OpenAIEmbeddings` + `ChromaDB` |
| **Fine-Tuning / Few-Shot** | Section 4 | In-context learning via agent prompt versions |
| **Pre-Trained LLM** | Section 5 | `gpt-5-mini` via OpenAI API |
| **Context-Specific LLM** | Section 5 | LangChain Agent with 3 tools |
| **LLM API** | Section 6 | `AgentExecutor` as the API interface |
| **End User Apps / Prompts** | Section 7 | Question answering across security research papers |
| **Model Versioning** | Section 10 | MLflow experiment tracking + artifact logging |
| **Model Caching** | Section 3 | Embedding cache (ChromaDB reuse) |
| **Model Monitoring** | Section 8 | MLflow Tracing + autolog + RAGAS evaluation |
| **RLHF** | Section 9 | Human feedback collection + MLflow logging |

# Conclusion

## Key Takeaways

1. **LLMOps extends MLOps** with prompt engineering, pre-trained model selection,
   and human feedback - but shares foundations like versioning, monitoring, and evaluation.

2. **Agents > RAG chains** for complex tasks: agents can search, summarize, and
   reason across multiple research papers - adapting their approach per question rather than following a fixed pipeline.

3. **MLflow 3.10.x autolog** captures agent traces automatically - every tool call,
   LLM invocation, and reasoning step is logged with zero manual instrumentation.

4. **Prefect orchestrates** the pipeline, making it reproducible and monitorable.
   Each task (load, embed, build agent, query, evaluate) is tracked independently.

5. **The LLMOps lifecycle is iterative**: build agent -> evaluate -> collect feedback ->
   improve prompts -> repeat. MLflow tracks every iteration for comparison.

6. **Multi-document RAG** enables cross-paper analysis - the agent can synthesize insights
   from risk assessments, attack taxonomies, safety frameworks, and guardrail architectures.

## Challenges Addressed

| LLMOps Challenge | Our Solution |
|---|---|
| Unstructured data | Multi-PDF loading + chunking + embeddings |
| Semantic drift | MLflow trace monitoring over time |
| Speed & latency | `max_iterations`, embedding caching |
| Cost efficiency | Agent decides which tools to use |
| Security risks | Error handling, iteration limits, content from security research |
| Model updates | Full config versioning in MLflow |

In [ ]:
# ============================================================================
# CLEANUP (Run this cell manually when you're done exploring the UIs)
# ============================================================================
# This cell is skipped during automated runs. Run it manually when done.
# ============================================================================

CLEANUP = False  # Set to True and run this cell manually to cleanup

if CLEANUP:
    for proc in [cloudflared_process, mlflow_process, prefect_process]:
        try:
            proc.terminate()
        except Exception:
            pass
    print("Cleanup complete.")
    print(f"All MLflow data persisted at: {mlruns_path}")
else:
    print("Skipping cleanup - UIs remain accessible.")
    print("Set CLEANUP = True and re-run this cell when you're done exploring.")